# Impact Analysis — Economic Impact Dashboard

What-if scenarios, causal analysis, and executive summary generation
for the environment-economy dashboard.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

## Analysis Steps

1. Load trained model and panel data
2. Define what-if scenarios (drought, flood, temperature increase)
3. Simulate economic impact under each scenario
4. Estimate confidence intervals via bootstrap
5. Generate executive summary with key findings

In [ ]:
# Load model and data
model = joblib.load('../models/ridge_panel.pkl')
panel = pd.read_parquet('../data/processed/panel_features.parquet')

target = 'gdp_index'
feature_cols = [c for c in panel.columns if c not in ['date', 'region', target]]
X_baseline = panel[feature_cols].values
y_baseline = model.predict(X_baseline)

In [ ]:
# What-if scenarios
scenarios = {
    'Drought (-30% NDVI)': {'ndvi_mean': -0.30, 'precipitation': -0.40},
    'Flood (+50% precip)': {'precipitation': 0.50, 'ndvi_mean': -0.10},
    'Warming (+2C)': {'temperature_avg': 2.0},
}

results = []
for name, adjustments in scenarios.items():
    X_scenario = panel[feature_cols].copy()
    for col, delta in adjustments.items():
        matching = [c for c in feature_cols if c.startswith(col)]
        for mc in matching:
            if 'lag' in mc or 'roll' in mc:
                X_scenario[mc] *= (1 + delta)
            else:
                X_scenario[mc] *= (1 + delta)
    y_scenario = model.predict(X_scenario.values)
    impact = ((y_scenario - y_baseline) / y_baseline * 100).mean()
    results.append({'scenario': name, 'avg_gdp_impact_pct': round(impact, 2)})
    print(f'{name}: avg GDP impact = {impact:+.2f}%')

results_df = pd.DataFrame(results)
print('\n--- Executive Summary ---')
print(results_df.to_string(index=False))